# Myerson's Lemma Truthfulness Analysis (k=25, Batch Averaging)

**Approach:** Generate 25 samples per bid combination. Group into 5 batches of 5.
For each batch, select the welfare-maximizing sample. Then average the alignment
scores across the 5 batch winners.

This approximates the **expected allocation** under welfare-optimal selection,
which is what risk-neutral bidders target. Compared to picking 1 welfare-max
out of 25 (high variance), this gives a much better estimate of E[alignment].

$$p_i(b_i) = b_i \cdot x_i(b_i) - \int_0^{b_i} x_i(z) \, dz$$

In [ ]:
import os
import json
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy import interpolate
from collections import defaultdict

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

RESULTS_DIR = '../results/truthfulness_25'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Batch configuration
NUM_BATCHES = 5
BATCH_SIZE = 5  # samples per batch
TOTAL_SAMPLES = NUM_BATCHES * BATCH_SIZE  # 25
print(f'Batch config: {NUM_BATCHES} batches x {BATCH_SIZE} samples = {TOTAL_SAMPLES} total')

## 1. Load Alignment Data

In [ ]:
def load_alignment_data(alignment_dir, prompts_path, prompt_indices=None):
    """Load all alignment JSONs from a directory.
    
    Args:
        alignment_dir: path to alignment directory
        prompts_path: path to prompts JSON
        prompt_indices: list of prompt indices to load (None = all)
    
    Returns a DataFrame with columns:
        prompt_idx, b1, b2, sample_idx, base_alignment,
        agent1_alignment, agent2_alignment, total_welfare, weighted_alignment
    """
    if not os.path.exists(alignment_dir):
        print(f'Alignment dir not found: {alignment_dir}')
        return pd.DataFrame()
    
    with open(prompts_path) as f:
        prompts = json.load(f)
    
    if prompt_indices is None:
        prompt_indices = list(range(len(prompts)))
    
    records = []
    for prompt_idx in prompt_indices:
        prompt_dir = os.path.join(alignment_dir, f'prompt_{prompt_idx:03d}')
        if not os.path.isdir(prompt_dir):
            print(f'  Skipping prompt {prompt_idx} (not found)')
            continue
        
        json_files = glob.glob(os.path.join(prompt_dir, 'alignment_*.json'))
        if not json_files:
            print(f'  Skipping prompt {prompt_idx} (no alignment files)')
            continue
        
        for json_file in json_files:
            with open(json_file) as f:
                data = json.load(f)
            
            meta = data['metadata']
            bids = meta['bids']
            scores = data['alignment_scores']
            welfare = data['welfare_metrics']
            
            records.append({
                'prompt_idx': meta['prompt_index'],
                'b1': bids[0],
                'b2': bids[1],
                'sample_idx': meta['sample_index'],
                'base_alignment': scores['base_alignment'],
                'agent1_alignment': scores['agent1_alignment'],
                'agent2_alignment': scores['agent2_alignment'],
                'total_welfare': welfare['total_welfare'],
                'weighted_alignment': welfare['weighted_alignment'],
                'base_prompt': prompts[meta['prompt_index']]['base_prompt'],
                'agent1_prompt': prompts[meta['prompt_index']]['agent1_prompt'],
                'agent2_prompt': prompts[meta['prompt_index']]['agent2_prompt'],
            })
    
    df = pd.DataFrame(records)
    if len(df) > 0:
        print(f'Loaded {len(df)} alignment records from {alignment_dir}')
        sample_counts = df.groupby(['prompt_idx', 'b1', 'b2']).size()
        print(f'  Prompts: {sorted(df.prompt_idx.unique())}')
        print(f'  Bid combos: {len(sample_counts)}')
        print(f'  Samples per combo: min={sample_counts.min()}, max={sample_counts.max()}, median={sample_counts.median()}')
    else:
        print(f'No alignment data found in {alignment_dir}')
    return df

In [ ]:
# Load data for prompts 2 and 3
df_regular = load_alignment_data(
    '../alignment/truthfulness_clip_regular',
    '../prompts/truthfulness_regular.json',
    prompt_indices=[2, 3]
)

HAS_DATA = len(df_regular) > 0
if HAS_DATA:
    # Show how many samples we have per combo
    sample_dist = df_regular.groupby(['prompt_idx', 'b1', 'b2']).size()
    print(f'\nSample count distribution:')
    print(sample_dist.value_counts().sort_index())
    n_complete = (sample_dist >= TOTAL_SAMPLES).sum()
    print(f'\nBid combos with >= {TOTAL_SAMPLES} samples: {n_complete}/{len(sample_dist)}')

## 2. Batch Welfare-Optimal Averaging

For each (prompt, bid_combo):
1. Take the 25 samples, sort by sample_idx
2. Split into 5 batches: [s0-s4], [s5-s9], [s10-s14], [s15-s19], [s20-s24]
3. In each batch, pick the sample with highest `total_welfare`
4. Average the alignment scores across the 5 batch winners

This gives us the expected allocation under welfare-optimal selection.

In [ ]:
def batch_welfare_average(df, num_batches=5, batch_size=5):
    """For each (prompt_idx, b1, b2), split samples into batches,
    pick welfare-optimal from each batch, and average alignments.
    
    Only includes bid combos with FULL k=25 data (all 5 batches).
    
    Returns DataFrame with one row per (prompt_idx, b1, b2) containing
    averaged alignment scores plus per-batch values for error bar computation.
    """
    if len(df) == 0:
        return df
    
    total_needed = num_batches * batch_size
    records = []
    skipped = 0
    
    for (pidx, b1, b2), group in df.groupby(['prompt_idx', 'b1', 'b2']):
        group_sorted = group.sort_values('sample_idx').reset_index(drop=True)
        
        if len(group_sorted) < total_needed:
            # Skip combos without full k=25 data
            skipped += 1
            continue
        else:
            n_complete_batches = num_batches
        
        batch_winners_a1 = []
        batch_winners_a2 = []
        batch_winners_base = []
        
        for batch_i in range(n_complete_batches):
            start = batch_i * batch_size
            end = start + batch_size
            batch = group_sorted.iloc[start:end]
            
            # Pick welfare-optimal within this batch
            best_idx = batch['total_welfare'].idxmax()
            winner = batch.loc[best_idx]
            
            batch_winners_a1.append(winner['agent1_alignment'])
            batch_winners_a2.append(winner['agent2_alignment'])
            batch_winners_base.append(winner['base_alignment'])
        
        records.append({
            'prompt_idx': pidx,
            'b1': b1,
            'b2': b2,
            'agent1_alignment': np.mean(batch_winners_a1),
            'agent2_alignment': np.mean(batch_winners_a2),
            'base_alignment': np.mean(batch_winners_base),
            'agent1_alignment_std': np.std(batch_winners_a1),
            'agent2_alignment_std': np.std(batch_winners_a2),
            'agent1_batch_vals': list(batch_winners_a1),
            'agent2_batch_vals': list(batch_winners_a2),
            'n_batches': n_complete_batches,
        })
    
    result = pd.DataFrame(records)
    print(f'Batch-averaged {len(result)} bid combos ({skipped} skipped for < {total_needed} samples)')
    if len(result) > 0:
        print(f'  Batches used: {result["n_batches"].value_counts().to_dict()}')
    return result

if HAS_DATA:
    df_avg = batch_welfare_average(df_regular, NUM_BATCHES, BATCH_SIZE)
    print(f'\nResult: {len(df_avg)} averaged allocation points')
    if len(df_avg) > 0:
        print(df_avg.head(10))
    else:
        print('No combos with full k=25 data yet')

## 3. Build Allocation Curves

In [ ]:
# Bid values
DISCRETE_BIDS = np.round(np.arange(0.0, 1.05, 0.1), 2)
FIXED_VALUES = [0.3, 0.5, 0.7]
INTERP_BIDS = np.round(np.arange(0.05, 1.0, 0.1), 2)
ALL_BIDS = np.sort(np.unique(np.concatenate([DISCRETE_BIDS, INTERP_BIDS])))

# True values for regret computation (0.1 increments)
REGRET_TRUE_VALUES = np.round(np.arange(0.1, 1.0, 0.1), 2)

print(f'All bids ({len(ALL_BIDS)}): {ALL_BIDS}')
print(f'Fixed opponent values: {FIXED_VALUES}')
print(f'Regret true values ({len(REGRET_TRUE_VALUES)}): {REGRET_TRUE_VALUES}')

In [ ]:
def build_allocation_curves(df_avg, agent_col, vary_col, fixed_col, min_points=5, bid_subset='all'):
    """Build allocation curves from batch-averaged data.
    
    Args:
        df_avg: batch-averaged DataFrame (only k=25 points)
        agent_col: 'agent1_alignment' or 'agent2_alignment'
        vary_col: 'b1' or 'b2' (the bid that varies)
        fixed_col: 'b2' or 'b1' (the fixed opponent bid)
        min_points: minimum number of bid points needed to form a curve
        bid_subset: 'all' = use all bid points (0.05 increments),
                    'discrete' = only 0.1-increment points (for interpolation comparison)
    
    Returns:
        dict: {(prompt_idx, fixed_val): {'bids': array, 'allocations': array, 
               'stds': array, 'interp_fn': callable, 'batch_interp_fns': list}}
    """
    curves = {}
    if len(df_avg) == 0:
        return curves
    
    std_col = agent_col + '_std'
    batch_col = agent_col.replace('_alignment', '_batch_vals')
    
    for prompt_idx in df_avg['prompt_idx'].unique():
        for fixed_val in FIXED_VALUES:
            mask = (df_avg['prompt_idx'] == prompt_idx) & (np.isclose(df_avg[fixed_col], fixed_val))
            subset = df_avg[mask].sort_values(vary_col)
            
            # Filter to discrete bids only if requested
            if bid_subset == 'discrete':
                discrete_mask = subset[vary_col].apply(
                    lambda x: any(np.isclose(x, d) for d in DISCRETE_BIDS))
                subset = subset[discrete_mask]
            
            if len(subset) < min_points:
                continue
            
            bids = subset[vary_col].values
            allocations = subset[agent_col].values
            stds = subset[std_col].values if std_col in subset.columns else np.zeros_like(allocations)
            
            interp_fn = interpolate.interp1d(
                bids, allocations, kind='linear',
                fill_value='extrapolate', bounds_error=False
            )
            
            # Build per-batch interpolation functions
            batch_interp_fns = []
            if batch_col in subset.columns:
                batch_vals_list = subset[batch_col].tolist()
                n_batches = len(batch_vals_list[0]) if batch_vals_list else 0
                for bi in range(n_batches):
                    batch_allocs = np.array([bv[bi] for bv in batch_vals_list])
                    batch_fn = interpolate.interp1d(
                        bids, batch_allocs, kind='linear',
                        fill_value='extrapolate', bounds_error=False
                    )
                    batch_interp_fns.append(batch_fn)
            
            curves[(prompt_idx, fixed_val)] = {
                'bids': bids,
                'allocations': allocations,
                'stds': stds,
                'interp_fn': interp_fn,
                'batch_interp_fns': batch_interp_fns,
            }
    
    return curves

if HAS_DATA and len(df_avg) > 0:
    # "Actual" curves: use all bid points (0.0, 0.05, 0.1, ..., 1.0)
    curves_a1 = build_allocation_curves(df_avg, 'agent1_alignment', 'b1', 'b2', bid_subset='all')
    curves_a2 = build_allocation_curves(df_avg, 'agent2_alignment', 'b2', 'b1', bid_subset='all')
    
    # "Interpolated" curves: use only 0.1-increment points, interpolate to 0.05
    curves_a1_interp = build_allocation_curves(df_avg, 'agent1_alignment', 'b1', 'b2', bid_subset='discrete')
    curves_a2_interp = build_allocation_curves(df_avg, 'agent2_alignment', 'b2', 'b1', bid_subset='discrete')
    
    print('=== Actual (all bid points) ===')
    print(f'Agent 1 curves: {len(curves_a1)}')
    for k in sorted(curves_a1.keys()):
        print(f'  {k}: {len(curves_a1[k]["bids"])} points, {len(curves_a1[k]["batch_interp_fns"])} batch fns')
    print(f'Agent 2 curves: {len(curves_a2)}')
    for k in sorted(curves_a2.keys()):
        print(f'  {k}: {len(curves_a2[k]["bids"])} points, {len(curves_a2[k]["batch_interp_fns"])} batch fns')
    
    print('\n=== Interpolated (0.1-increment points only) ===')
    print(f'Agent 1 curves: {len(curves_a1_interp)}')
    for k in sorted(curves_a1_interp.keys()):
        print(f'  {k}: {len(curves_a1_interp[k]["bids"])} points')
    print(f'Agent 2 curves: {len(curves_a2_interp)}')
    for k in sorted(curves_a2_interp.keys()):
        print(f'  {k}: {len(curves_a2_interp[k]["bids"])} points')
else:
    curves_a1, curves_a2 = {}, {}
    curves_a1_interp, curves_a2_interp = {}, {}
    print('No k=25 data to build curves')

## 4. Myerson Payments & Utilities

In [ ]:
def compute_myerson_payment(interp_fn, bid, n_quadrature=1000):
    """p(b) = b * x(b) - integral_0^b x(z) dz"""
    x_at_bid = float(interp_fn(bid))
    z_vals = np.linspace(0, bid, n_quadrature)
    x_vals = interp_fn(z_vals)
    integral = np.trapz(x_vals, z_vals)
    return max(bid * x_at_bid - integral, 0.0)


def compute_utility(true_value, bid, interp_fn):
    """u(v, b) = v * x(b) - p(b), where both allocation and payment use the same curve."""
    x_at_bid = float(interp_fn(bid))
    payment = compute_myerson_payment(interp_fn, bid)
    return true_value * x_at_bid - payment


def compute_utility_mixed(true_value, bid, alloc_fn, pricing_fn):
    """u(v, b) = v * x_actual(b) - p_pricing(b).
    
    Uses actual allocation for value, but payment from a (possibly coarser) pricing curve.
    This models: mechanism sets prices from 0.1-increment data,
    but agent gets the actual allocation from 0.05-increment images.
    """
    x_actual = float(alloc_fn(bid))
    payment = compute_myerson_payment(pricing_fn, bid)
    return true_value * x_actual - payment


def compute_regret_curve(interp_fn, true_values=None, bid_grid=None):
    """Compute relative regret for each true value (single curve for both alloc and payment)."""
    if true_values is None:
        true_values = REGRET_TRUE_VALUES
    if bid_grid is None:
        bid_grid = ALL_BIDS
    
    results = []
    for v in true_values:
        u_truthful = compute_utility(v, v, interp_fn)
        utilities = [compute_utility(v, b, interp_fn) for b in bid_grid]
        best_idx = np.argmax(utilities)
        u_optimal = utilities[best_idx]
        b_optimal = bid_grid[best_idx]
        
        if abs(u_truthful) > 1e-10:
            rel_regret = (u_optimal - u_truthful) / abs(u_truthful)
        else:
            rel_regret = 0.0 if abs(u_optimal - u_truthful) < 1e-10 else np.inf
        
        results.append({
            'true_value': v,
            'truthful_utility': u_truthful,
            'optimal_utility': u_optimal,
            'optimal_bid': b_optimal,
            'relative_regret': rel_regret,
        })
    
    return pd.DataFrame(results)


def compute_regret_curve_mixed(alloc_fn, pricing_fn, true_values=None, bid_grid=None):
    """Compute relative regret using actual allocation but interpolated pricing.
    
    Agent utility: u(v, b) = v * x_actual(b) - p_pricing(b)
    This tests: if the mechanism sets Myerson prices from a coarser grid (0.1 increments),
    is truthful bidding still optimal when agents face the actual allocation (0.05 increments)?
    """
    if true_values is None:
        true_values = REGRET_TRUE_VALUES
    if bid_grid is None:
        bid_grid = ALL_BIDS
    
    results = []
    for v in true_values:
        u_truthful = compute_utility_mixed(v, v, alloc_fn, pricing_fn)
        utilities = [compute_utility_mixed(v, b, alloc_fn, pricing_fn) for b in bid_grid]
        best_idx = np.argmax(utilities)
        u_optimal = utilities[best_idx]
        b_optimal = bid_grid[best_idx]
        
        if abs(u_truthful) > 1e-10:
            rel_regret = (u_optimal - u_truthful) / abs(u_truthful)
        else:
            rel_regret = 0.0 if abs(u_optimal - u_truthful) < 1e-10 else np.inf
        
        results.append({
            'true_value': v,
            'truthful_utility': u_truthful,
            'optimal_utility': u_optimal,
            'optimal_bid': b_optimal,
            'relative_regret': rel_regret,
        })
    
    return pd.DataFrame(results)


# Test
if curves_a1:
    test_key = list(curves_a1.keys())[0]
    print('=== Single-curve regret (actual) ===')
    test_regret = compute_regret_curve(curves_a1[test_key]['interp_fn'])
    print(f'Test curve: {test_key}')
    print(test_regret[['true_value', 'truthful_utility', 'optimal_bid', 'relative_regret']].to_string(index=False))
    
    if test_key in curves_a1_interp:
        print('\n=== Mixed regret (actual alloc + interpolated pricing) ===')
        test_mixed = compute_regret_curve_mixed(
            curves_a1[test_key]['interp_fn'],       # actual allocation (all points)
            curves_a1_interp[test_key]['interp_fn']  # pricing from 0.1-only
        )
        print(test_mixed[['true_value', 'truthful_utility', 'optimal_bid', 'relative_regret']].to_string(index=False))

## 5. Allocation Curve Plots

In [ ]:
def plot_allocation_curves(curves, agent_name, prompts):
    """Plot allocation curves with error bars from batch std."""
    if not curves:
        print(f'No curves available for {agent_name}')
        return None
    
    prompt_indices = sorted(set(k[0] for k in curves.keys()))
    colors = {0.3: 'tab:blue', 0.5: 'tab:orange', 0.7: 'tab:red'}
    
    fig, axes = plt.subplots(1, len(prompt_indices), figsize=(6*len(prompt_indices), 5), squeeze=False)
    
    for col, pidx in enumerate(prompt_indices):
        ax = axes[0, col]
        for fv in FIXED_VALUES:
            key = (pidx, fv)
            if key not in curves:
                continue
            c = curves[key]
            
            # Plot data points with error bars
            ax.errorbar(c['bids'], c['allocations'], yerr=c['stds'],
                       fmt='o', color=colors[fv], markersize=4, capsize=2,
                       alpha=0.6, zorder=5)
            
            # Plot interpolated line
            fine_bids = np.linspace(0, 1, 200)
            ax.plot(fine_bids, c['interp_fn'](fine_bids), color=colors[fv],
                    label=f'Opponent={fv}', linewidth=2)
        
        ax.set_xlabel(f'{agent_name} Bid')
        ax.set_ylabel('CLIP Alignment (batch-averaged)')
        ax.set_title(f'P{pidx}: {prompts[pidx]["base_prompt"][:35]}...', fontsize=10)
        ax.legend(fontsize=9)
        ax.set_xlim(-0.02, 1.02)
    
    fig.suptitle(f'Allocation Curves — {agent_name} (k=25, 5 batches of 5)', fontsize=14)
    plt.tight_layout()
    return fig

with open('../prompts/truthfulness_regular.json') as f:
    prompts_regular = json.load(f)

if curves_a1:
    fig = plot_allocation_curves(curves_a1, 'Agent 1', prompts_regular)
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'allocation_curves_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()

if curves_a2:
    fig = plot_allocation_curves(curves_a2, 'Agent 2', prompts_regular)
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'allocation_curves_a2.png'), dpi=150, bbox_inches='tight')
    plt.show()

if not curves_a1 and not curves_a2:
    print('No k=25 curves available yet. Need k=25 generation + alignment first.')

## 6. Mean Allocation Curves (across prompts)

In [ ]:
def plot_mean_allocation_curves(curves, agent_name):
    """Plot mean allocation curves averaged across prompts with std bands."""
    if not curves:
        print(f'No curves for mean plot ({agent_name})')
        return None
    
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = {0.3: 'tab:blue', 0.5: 'tab:orange', 0.7: 'tab:red'}
    
    for fv in FIXED_VALUES:
        prompt_keys = [k for k in curves if k[1] == fv]
        if not prompt_keys:
            continue
        
        fine_bids = np.linspace(0, 1, 200)
        all_allocs = np.array([curves[k]['interp_fn'](fine_bids) for k in prompt_keys])
        mean_alloc = all_allocs.mean(axis=0)
        std_alloc = all_allocs.std(axis=0)
        
        ax.plot(fine_bids, mean_alloc, color=colors[fv], label=f'Opponent={fv}', linewidth=2)
        ax.fill_between(fine_bids, mean_alloc - std_alloc, mean_alloc + std_alloc,
                        color=colors[fv], alpha=0.15)
    
    ax.set_xlabel(f'{agent_name} Bid')
    ax.set_ylabel('CLIP Alignment (mean across prompts)')
    ax.set_title(f'Mean Allocation Curve — {agent_name} (k=25, batch-averaged)')
    ax.legend()
    ax.set_xlim(-0.02, 1.02)
    plt.tight_layout()
    return fig

if curves_a1:
    fig = plot_mean_allocation_curves(curves_a1, 'Agent 1')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'allocation_curves_mean_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()

if curves_a2:
    fig = plot_mean_allocation_curves(curves_a2, 'Agent 2')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'allocation_curves_mean_a2.png'), dpi=150, bbox_inches='tight')
    plt.show()

if not curves_a1 and not curves_a2:
    print('No curves for mean allocation plot')

In [ ]:
def plot_single_agent_allocation_mean(curves, agent_name, fixed_val=0.3, color='tab:blue'):
    """Plot one agent's mean allocation curve (opponent fixed at fixed_val)."""
    fine_bids = np.linspace(0, 1, 200)
    keys = [k for k in curves if k[1] == fixed_val]
    if not keys:
        print(f'No curves for {agent_name} at fixed_val={fixed_val}')
        return None

    all_allocs = np.array([curves[k]['interp_fn'](fine_bids) for k in keys])
    mean_alloc = all_allocs.mean(axis=0)
    std_alloc = all_allocs.std(axis=0)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(fine_bids, mean_alloc, color=color, linewidth=2.5)
    ax.fill_between(fine_bids, mean_alloc - std_alloc, mean_alloc + std_alloc,
                    color=color, alpha=0.15)
    ax.set_xlabel('Agent Bid', fontsize=13)
    ax.set_ylabel('CLIP Alignment', fontsize=13)
    ax.set_title(f'Expected Allocation Curve ({agent_name})', fontsize=15)
    ax.set_xlim(-0.02, 1.02)
    plt.tight_layout()
    return fig

if curves_a1:
    fig = plot_single_agent_allocation_mean(curves_a1, 'Agent 1', fixed_val=0.3, color='tab:blue')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'allocation_curves_mean_a1_opp0.3.png'),
                    dpi=150, bbox_inches='tight')
    plt.show()

if curves_a2:
    fig = plot_single_agent_allocation_mean(curves_a2, 'Agent 2', fixed_val=0.3, color='tab:red')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'allocation_curves_mean_a2_opp0.3.png'),
                    dpi=150, bbox_inches='tight')
    plt.show()

## 7. Payment Curves

In [ ]:
def plot_payment_curves(curves, agent_name, prompts):
    """Plot Myerson payment curves."""
    prompt_indices = sorted(set(k[0] for k in curves.keys()))
    colors = {0.3: 'tab:blue', 0.5: 'tab:orange', 0.7: 'tab:red'}
    
    fig, axes = plt.subplots(1, len(prompt_indices), figsize=(6*len(prompt_indices), 5), squeeze=False)
    
    for col, pidx in enumerate(prompt_indices):
        ax = axes[0, col]
        for fv in FIXED_VALUES:
            key = (pidx, fv)
            if key not in curves:
                continue
            interp_fn = curves[key]['interp_fn']
            fine_bids = np.linspace(0.01, 1.0, 200)
            payments = [compute_myerson_payment(interp_fn, b) for b in fine_bids]
            ax.plot(fine_bids, payments, color=colors[fv], label=f'Opponent={fv}', linewidth=2)
        
        ax.set_xlabel(f'{agent_name} Bid')
        ax.set_ylabel('Myerson Payment')
        ax.set_title(f'P{pidx}: {prompts[pidx]["base_prompt"][:35]}...', fontsize=10)
        ax.legend(fontsize=9)
    
    fig.suptitle(f'Payment Curves — {agent_name} (k=25, batch-averaged)', fontsize=14)
    plt.tight_layout()
    return fig

if curves_a1:
    fig = plot_payment_curves(curves_a1, 'Agent 1', prompts_regular)
    fig.savefig(os.path.join(RESULTS_DIR, 'payment_curves_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()

if curves_a2:
    fig = plot_payment_curves(curves_a2, 'Agent 2', prompts_regular)
    fig.savefig(os.path.join(RESULTS_DIR, 'payment_curves_a2.png'), dpi=150, bbox_inches='tight')
    plt.show()

if not curves_a1 and not curves_a2:
    print('No curves available for payment plots')

## 8. Utility Curves

In [ ]:
def plot_utility_curves(curves, agent_name, prompts):
    """Plot utility curves u_i(b_i) for different true values."""
    prompt_indices = sorted(set(k[0] for k in curves.keys()))
    available_fvs = sorted(set(k[1] for k in curves.keys()))
    true_values_to_plot = [0.15, 0.35, 0.55, 0.75, 0.95]
    cmap = cm.viridis
    
    n_rows = len(available_fvs) if available_fvs else 1
    n_cols = len(prompt_indices) if prompt_indices else 1
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(6*n_cols, 4*n_rows), squeeze=False)
    
    for row, fv in enumerate(available_fvs):
        for col, pidx in enumerate(prompt_indices):
            ax = axes[row, col]
            key = (pidx, fv)
            if key not in curves:
                ax.set_visible(False)
                continue
            
            interp_fn = curves[key]['interp_fn']
            bid_grid = np.linspace(0.01, 1.0, 200)
            
            for i, v in enumerate(true_values_to_plot):
                color = cmap(i / (len(true_values_to_plot) - 1))
                utilities = [compute_utility(v, b, interp_fn) for b in bid_grid]
                ax.plot(bid_grid, utilities, color=color, label=f'v={v}', linewidth=1.5)
                u_truthful = compute_utility(v, v, interp_fn)
                ax.axvline(v, color=color, linestyle=':', alpha=0.3, linewidth=0.8)
                ax.plot(v, u_truthful, 'o', color=color, markersize=6)
            
            ax.set_xlabel(f'{agent_name} Bid')
            if row == 0:
                ax.set_title(f'P{pidx}: {prompts[pidx]["base_prompt"][:30]}...', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'Opp={fv}\nUtility')
            ax.legend(fontsize=7, ncol=2)
    
    fig.suptitle(f'Utility Curves — {agent_name} (k=25, batch-averaged)', fontsize=14)
    plt.tight_layout()
    return fig

if curves_a1:
    fig = plot_utility_curves(curves_a1, 'Agent 1', prompts_regular)
    fig.savefig(os.path.join(RESULTS_DIR, 'utility_curves_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()

if curves_a2:
    fig = plot_utility_curves(curves_a2, 'Agent 2', prompts_regular)
    fig.savefig(os.path.join(RESULTS_DIR, 'utility_curves_a2.png'), dpi=150, bbox_inches='tight')
    plt.show()

if not curves_a1 and not curves_a2:
    print('No curves available for utility plots')

## 9. Regret Heatmaps

In [ ]:
def compute_regret_heatmap(curves):
    """Compute regret for all (prompt, fixed_val, true_value)."""
    if not curves:
        return pd.DataFrame()
    all_regrets = []
    for (pidx, fv), curve_data in curves.items():
        regret_df = compute_regret_curve(curve_data['interp_fn'])
        regret_df['prompt_idx'] = pidx
        regret_df['fixed_val'] = fv
        all_regrets.append(regret_df)
    return pd.concat(all_regrets, ignore_index=True)


def plot_regret_heatmap(regret_df, agent_name):
    """Plot regret heatmap averaged across prompts."""
    if len(regret_df) == 0:
        print(f'No regret data for {agent_name}')
        return None
    mean_regret = regret_df.groupby(['fixed_val', 'true_value'])['relative_regret'].mean().reset_index()
    pivot = mean_regret.pivot(index='fixed_val', columns='true_value', values='relative_regret')
    
    fig, ax = plt.subplots(figsize=(12, max(2, 1.5*len(pivot.index))))
    im = ax.imshow(pivot.values, cmap='RdYlGn_r', aspect='auto', vmin=0,
                   vmax=max(0.01, pivot.values.max()))
    
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{v:.2f}' for v in pivot.columns], rotation=45)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{v:.1f}' for v in pivot.index])
    ax.set_xlabel('Agent True Value')
    ax.set_ylabel('Opponent Fixed Bid')
    ax.set_title(f'Mean Relative Regret — {agent_name} (k=25, batch-averaged)')
    
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.4f}', ha='center', va='center', fontsize=8,
                   color='white' if val > pivot.values.max()/2 else 'black')
    
    plt.colorbar(im, ax=ax, label='Relative Regret')
    plt.tight_layout()
    return fig

regret_a1 = compute_regret_heatmap(curves_a1) if curves_a1 else pd.DataFrame()
regret_a2 = compute_regret_heatmap(curves_a2) if curves_a2 else pd.DataFrame()

if len(regret_a1) > 0:
    print(f'Agent 1 — Mean regret: {regret_a1["relative_regret"].mean():.4f}, Max: {regret_a1["relative_regret"].max():.4f}')
    fig = plot_regret_heatmap(regret_a1, 'Agent 1')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'regret_heatmap_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()

if len(regret_a2) > 0:
    print(f'Agent 2 — Mean regret: {regret_a2["relative_regret"].mean():.4f}, Max: {regret_a2["relative_regret"].max():.4f}')
    fig = plot_regret_heatmap(regret_a2, 'Agent 2')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'regret_heatmap_a2.png'), dpi=150, bbox_inches='tight')
    plt.show()

if len(regret_a1) == 0 and len(regret_a2) == 0:
    print('No curves available for regret analysis')

## 9b. Per-Prompt Regret Line Plots

In [ ]:
def _plot_allocation_on_ax(ax, interp_fn, bids, allocations, stds, color, label,
                           pricing_fn=None):
    """Helper: plot allocation curve on a given axis."""
    fine_bids = np.linspace(0, 1, 200)
    ax.plot(fine_bids, interp_fn(fine_bids), color=color, linewidth=2, label=label)
    ax.errorbar(bids, allocations, yerr=stds, fmt='o', color=color,
                markersize=3, capsize=2, alpha=0.5, zorder=5)
    if pricing_fn is not None:
        ax.plot(fine_bids, pricing_fn(fine_bids), color=color, linewidth=1.5,
                linestyle='--', alpha=0.6, label=f'{label} (pricing)')


def plot_regret_mean(curves, agent_name, prompts, subtitle='',
                     pricing_curves=None):
    """Plot allocation + regret per prompt. Top row = allocation, bottom row = regret."""
    if not curves:
        print(f'No curves for regret mean plot ({agent_name})')
        return None
    
    prompt_indices = sorted(set(k[0] for k in curves.keys()))
    available_fvs = sorted(set(k[1] for k in curves.keys()))
    colors = {0.3: 'tab:blue', 0.5: 'tab:orange', 0.7: 'tab:red'}
    markers = {0.3: 'o', 0.5: 's', 0.7: '^'}
    fine_tv = np.round(np.arange(0.05, 1.0, 0.05), 2)
    
    fig, axes = plt.subplots(2, len(prompt_indices),
                             figsize=(7*len(prompt_indices), 9), squeeze=False)
    
    for col, pidx in enumerate(prompt_indices):
        ax_alloc = axes[0, col]
        ax_regret = axes[1, col]
        
        for fv in available_fvs:
            key = (pidx, fv)
            if key not in curves:
                continue
            c = curves[key]
            color = colors.get(fv, 'tab:gray')
            
            # Allocation curve
            pfn = pricing_curves[key]['interp_fn'] if (pricing_curves and key in pricing_curves) else None
            _plot_allocation_on_ax(ax_alloc, c['interp_fn'], c['bids'],
                                   c['allocations'], c['stds'], color,
                                   f'Opp={fv}', pricing_fn=pfn)
            
            # Regret curve
            if pricing_curves and key in pricing_curves:
                rdf = compute_regret_curve_mixed(
                    c['interp_fn'], pricing_curves[key]['interp_fn'],
                    true_values=fine_tv)
            else:
                rdf = compute_regret_curve(c['interp_fn'], true_values=fine_tv)
            ax_regret.plot(rdf['true_value'], rdf['relative_regret'],
                           marker=markers.get(fv, 'o'), color=color,
                           label=f'Opponent={fv}', linewidth=2, markersize=4)
        
        ax_alloc.set_xlabel(f'{agent_name} Bid')
        ax_alloc.set_ylabel('CLIP Alignment')
        ax_alloc.set_title(f'P{pidx}: {prompts[pidx]["base_prompt"][:35]}...', fontsize=10)
        ax_alloc.legend(fontsize=8)
        ax_alloc.set_xlim(-0.02, 1.02)
        
        ax_regret.set_xlabel('Agent True Value')
        ax_regret.set_ylabel('Relative Regret')
        ax_regret.legend(fontsize=9)
        ax_regret.set_xlim(0.0, 1.0)
        ax_regret.set_ylim(-0.001, 0.05)
        ax_regret.axhline(0, color='gray', linewidth=0.5, linestyle='-')
    
    title = f'{agent_name} — Allocation & Regret (k=25, batch-averaged)'
    if subtitle:
        title += f'\n{subtitle}'
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    return fig


def plot_regret_mean_across_prompts(curves, agent_name, subtitle='',
                                     pricing_curves=None):
    """Plot allocation + regret averaged across prompts. Left = allocation, right = regret."""
    if not curves:
        print(f'No curves for mean regret ({agent_name})')
        return None
    
    available_fvs = sorted(set(k[1] for k in curves.keys()))
    colors = {0.3: 'tab:blue', 0.5: 'tab:orange', 0.7: 'tab:red'}
    markers = {0.3: 'o', 0.5: 's', 0.7: '^'}
    fine_tv = np.round(np.arange(0.05, 1.0, 0.05), 2)
    fine_bids = np.linspace(0, 1, 200)
    
    fig, (ax_alloc, ax_regret) = plt.subplots(1, 2, figsize=(14, 5))
    
    for fv in available_fvs:
        prompt_keys = [k for k in curves if k[1] == fv]
        if not prompt_keys:
            continue
        color = colors.get(fv, 'tab:gray')
        
        # Mean allocation
        all_allocs = np.array([curves[k]['interp_fn'](fine_bids) for k in prompt_keys])
        mean_alloc = all_allocs.mean(axis=0)
        ax_alloc.plot(fine_bids, mean_alloc, color=color, linewidth=2,
                      label=f'Opp={fv}')
        if pricing_curves:
            pricing_keys = [k for k in prompt_keys if k in pricing_curves]
            if pricing_keys:
                all_pricing = np.array([pricing_curves[k]['interp_fn'](fine_bids)
                                        for k in pricing_keys])
                ax_alloc.plot(fine_bids, all_pricing.mean(axis=0), color=color,
                              linewidth=1.5, linestyle='--', alpha=0.6,
                              label=f'Opp={fv} (pricing)')
        
        # Mean regret
        all_regrets = []
        for key in prompt_keys:
            if pricing_curves and key in pricing_curves:
                rdf = compute_regret_curve_mixed(
                    curves[key]['interp_fn'],
                    pricing_curves[key]['interp_fn'],
                    true_values=fine_tv)
            else:
                rdf = compute_regret_curve(curves[key]['interp_fn'], true_values=fine_tv)
            all_regrets.append(rdf['relative_regret'].values)
        mean_regret = np.mean(all_regrets, axis=0)
        ax_regret.plot(fine_tv, mean_regret, marker=markers.get(fv, 'o'),
                       color=color, label=f'Opponent={fv}', linewidth=2, markersize=4)
    
    ax_alloc.set_xlabel(f'{agent_name} Bid')
    ax_alloc.set_ylabel('CLIP Alignment (mean across prompts)')
    ax_alloc.set_title('Allocation Curves')
    ax_alloc.legend(fontsize=9)
    ax_alloc.set_xlim(-0.02, 1.02)
    
    ax_regret.set_xlabel('Agent True Value')
    ax_regret.set_ylabel('Relative Regret')
    ax_regret.set_title('Relative Regret')
    ax_regret.legend(fontsize=10)
    ax_regret.set_xlim(0.0, 1.0)
    ax_regret.set_ylim(-0.001, 0.05)
    ax_regret.axhline(0, color='gray', linewidth=0.5, linestyle='-')
    
    title = f'{agent_name} — Mean Allocation & Regret (k=25, batch-averaged)'
    if subtitle:
        title += f'\n{subtitle}'
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    return fig


def plot_regret_per_batch(curves, agent_name, prompts, subtitle='',
                          pricing_curves=None):
    """Plot per-batch allocation + regret. For each (fv, prompt):
    top = allocation (5 batch lines + mean), bottom = regret (5 batch lines + mean).
    Unrestricted y-axis on regret."""
    if not curves:
        print(f'No curves for per-batch regret ({agent_name})')
        return None
    
    prompt_indices = sorted(set(k[0] for k in curves.keys()))
    available_fvs = sorted(set(k[1] for k in curves.keys()))
    fine_tv = np.round(np.arange(0.05, 1.0, 0.05), 2)
    fine_bids = np.linspace(0, 1, 200)
    batch_colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
    
    n_fvs = len(available_fvs) if available_fvs else 1
    n_cols = len(prompt_indices) if prompt_indices else 1
    n_rows = 2 * n_fvs  # allocation + regret for each fv
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(7*n_cols, 4*n_rows), squeeze=False)
    
    for fv_idx, fv in enumerate(available_fvs):
        row_alloc = 2 * fv_idx
        row_regret = 2 * fv_idx + 1
        
        for col, pidx in enumerate(prompt_indices):
            ax_a = axes[row_alloc, col]
            ax_r = axes[row_regret, col]
            key = (pidx, fv)
            if key not in curves:
                ax_a.set_visible(False)
                ax_r.set_visible(False)
                continue
            
            curve_data = curves[key]
            batch_fns = curve_data.get('batch_interp_fns', [])
            pricing_batch_fns = None
            if pricing_curves and key in pricing_curves:
                pricing_batch_fns = pricing_curves[key].get('batch_interp_fns', [])
            
            for bi, bfn in enumerate(batch_fns):
                bc = batch_colors[bi % len(batch_colors)]
                
                # Allocation
                ax_a.plot(fine_bids, bfn(fine_bids), color=bc,
                          linewidth=1, alpha=0.5, label=f'Batch {bi+1}')
                
                # Regret
                if pricing_batch_fns and bi < len(pricing_batch_fns):
                    rdf = compute_regret_curve_mixed(bfn, pricing_batch_fns[bi],
                                                      true_values=fine_tv)
                else:
                    rdf = compute_regret_curve(bfn, true_values=fine_tv)
                ax_r.plot(rdf['true_value'], rdf['relative_regret'],
                          color=bc, linewidth=1, alpha=0.6, label=f'Batch {bi+1}')
            
            # Mean allocation
            ax_a.plot(fine_bids, curve_data['interp_fn'](fine_bids),
                      color='black', linewidth=2.5, label='Mean', zorder=10)
            if pricing_curves and key in pricing_curves:
                ax_a.plot(fine_bids, pricing_curves[key]['interp_fn'](fine_bids),
                          color='black', linewidth=2, linestyle='--', alpha=0.6,
                          label='Pricing', zorder=10)
            
            # Mean regret
            if pricing_curves and key in pricing_curves:
                rdf_mean = compute_regret_curve_mixed(
                    curve_data['interp_fn'],
                    pricing_curves[key]['interp_fn'],
                    true_values=fine_tv)
            else:
                rdf_mean = compute_regret_curve(curve_data['interp_fn'],
                                                 true_values=fine_tv)
            ax_r.plot(rdf_mean['true_value'], rdf_mean['relative_regret'],
                      color='black', linewidth=2.5, label='Mean', zorder=10)
            
            # Labels
            ax_a.set_xlim(-0.02, 1.02)
            ax_r.set_xlim(0.0, 1.0)
            ax_r.axhline(0, color='gray', linewidth=0.5, linestyle='-')
            
            if col == 0:
                ax_a.set_ylabel(f'Opp={fv}\nCLIP Alignment')
                ax_r.set_ylabel(f'Opp={fv}\nRelative Regret')
            if fv_idx == 0:
                ax_a.set_title(f'P{pidx}: {prompts[pidx]["base_prompt"][:35]}...', fontsize=10)
            if fv_idx == len(available_fvs) - 1:
                ax_r.set_xlabel('Agent True Value')
            ax_a.legend(fontsize=6, ncol=4)
            ax_r.legend(fontsize=6, ncol=4)
    
    title = f'Per-Batch Allocation & Regret — {agent_name} (k=25)'
    if subtitle:
        title += f'\n{subtitle}'
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    return fig


# ============================================================
# Plot three versions
# ============================================================

for agent_name, c_actual, c_interp, suffix in [
    ('Agent 1', curves_a1, curves_a1_interp, 'a1'),
    ('Agent 2', curves_a2, curves_a2_interp, 'a2'),
]:
    if not c_actual and not c_interp:
        continue
    
    # --- 1. ACTUAL ---
    if c_actual:
        fig = plot_regret_mean(c_actual, agent_name, prompts_regular,
                               subtitle='Actual allocation & payment (all bid points)')
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_mean_actual_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        
        fig = plot_regret_mean_across_prompts(c_actual, agent_name,
                               subtitle='Actual allocation & payment (all bid points)')
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_mean_across_actual_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        
        fig = plot_regret_per_batch(c_actual, agent_name, prompts_regular,
                                     subtitle='Actual allocation & payment (all bid points)')
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_batch_actual_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
    
    # --- 2. MIXED (actual alloc + interpolated pricing) ---
    if c_actual and c_interp:
        fig = plot_regret_mean(c_actual, agent_name, prompts_regular,
                               subtitle='Actual allocation + interpolated pricing (0.1-only)',
                               pricing_curves=c_interp)
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_mean_mixed_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        
        fig = plot_regret_mean_across_prompts(c_actual, agent_name,
                               subtitle='Actual allocation + interpolated pricing (0.1-only)',
                               pricing_curves=c_interp)
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_mean_across_mixed_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        
        fig = plot_regret_per_batch(c_actual, agent_name, prompts_regular,
                               subtitle='Actual allocation + interpolated pricing (0.1-only)',
                               pricing_curves=c_interp)
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_batch_mixed_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
    
    # --- 3. FULLY INTERPOLATED ---
    if c_interp:
        fig = plot_regret_mean(c_interp, agent_name, prompts_regular,
                               subtitle='Fully interpolated (0.1-only allocation & payment)')
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_mean_interp_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        
        fig = plot_regret_mean_across_prompts(c_interp, agent_name,
                               subtitle='Fully interpolated (0.1-only allocation & payment)')
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_mean_across_interp_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()
        
        fig = plot_regret_per_batch(c_interp, agent_name, prompts_regular,
                               subtitle='Fully interpolated (0.1-only allocation & payment)')
        fig.savefig(os.path.join(RESULTS_DIR, f'regret_batch_interp_{suffix}.png'),
                    dpi=150, bbox_inches='tight')
        plt.show()

if not curves_a1 and not curves_a2 and not curves_a1_interp and not curves_a2_interp:
    print('No curves available for regret line plots')

## 10. Monotonicity Verification

In [ ]:
def verify_monotonicity(curves, agent_name):
    """Verify allocation curves are monotonically non-decreasing."""
    if not curves:
        return None, pd.DataFrame()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    violations = []
    for key, curve_data in sorted(curves.items()):
        pidx, fv = key
        allocs = curve_data['allocations']
        diffs = np.diff(allocs)
        n_violations = np.sum(diffs < -0.001)  # small tolerance
        violation_magnitude = -np.minimum(diffs, 0).sum()
        violations.append({
            'prompt_idx': pidx, 'fixed_val': fv,
            'n_violations': n_violations,
            'violation_magnitude': violation_magnitude,
            'label': f'P{pidx},opp={fv}'
        })
    
    vdf = pd.DataFrame(violations)
    bar_colors = ['tab:green' if v == 0 else 'tab:red' for v in vdf['n_violations']]
    ax.bar(range(len(vdf)), vdf['violation_magnitude'], color=bar_colors)
    ax.set_xticks(range(len(vdf)))
    ax.set_xticklabels(vdf['label'], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Total Violation Magnitude')
    ax.set_title(f'Bid Monotonicity — {agent_name} (k=25, batch-averaged)')
    n_monotone = (vdf['n_violations'] == 0).sum()
    ax.text(0.95, 0.95, f'{n_monotone}/{len(vdf)} curves monotone',
            transform=ax.transAxes, ha='right', va='top', fontsize=12,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    plt.tight_layout()
    return fig, vdf

if curves_a1:
    fig, vdf = verify_monotonicity(curves_a1, 'Agent 1')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'bid_monotonicity_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(vdf.to_string(index=False))

## 11. Surplus Decomposition

In [ ]:
def plot_surplus_decomposition(curves, agent_name):
    """Plot surplus decomposition: agent utility + revenue = total value."""
    available_fvs = sorted(set(k[1] for k in curves.keys()))
    if not available_fvs:
        print(f'No curves for surplus decomposition ({agent_name})')
        return None
    
    fig, axes = plt.subplots(1, len(available_fvs), figsize=(6*len(available_fvs), 5), squeeze=False)
    
    for col, fv in enumerate(available_fvs):
        ax = axes[0, col]
        prompt_keys = [k for k in curves if k[1] == fv]
        if not prompt_keys:
            continue
        
        true_values = np.linspace(0.05, 0.95, 50)
        all_utilities, all_revenues, all_totals = [], [], []
        
        for key in prompt_keys:
            interp_fn = curves[key]['interp_fn']
            utils, revs, totals = [], [], []
            for v in true_values:
                x_v = float(interp_fn(v))
                payment = compute_myerson_payment(interp_fn, v)
                utils.append(v * x_v - payment)
                revs.append(payment)
                totals.append(v * x_v)
            all_utilities.append(utils)
            all_revenues.append(revs)
            all_totals.append(totals)
        
        mean_util = np.mean(all_utilities, axis=0)
        mean_rev = np.mean(all_revenues, axis=0)
        mean_total = np.mean(all_totals, axis=0)
        
        ax.fill_between(true_values, 0, mean_util, alpha=0.4, label='Agent Utility', color='tab:blue')
        ax.fill_between(true_values, mean_util, mean_util + mean_rev, alpha=0.4, label='Revenue', color='tab:orange')
        ax.plot(true_values, mean_total, 'k--', linewidth=1.5, label='Total Value v*x(v)')
        ax.set_xlabel('True Value (= Bid)')
        ax.set_ylabel('Value')
        ax.set_title(f'Opponent Bid = {fv}')
        ax.legend(fontsize=9)
    
    fig.suptitle(f'Surplus Decomposition — {agent_name} (k=25, batch-averaged)', fontsize=14)
    plt.tight_layout()
    return fig

if curves_a1:
    fig = plot_surplus_decomposition(curves_a1, 'Agent 1')
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'surplus_decomposition_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No Agent 1 curves for surplus decomposition')

## 12. Comparison: k=25 Batch-Averaged vs k=5 Single Welfare-Optimal

In [ ]:
def compare_methods(df, prompts, agent_col, vary_col, fixed_col, agent_name, k25_curves):
    """Compare batch-averaged (k=25) vs single welfare-optimal (k=5) vs single welfare-optimal (k=25).
    Only show rows for fixed values that have k=25 curves."""
    prompt_indices = sorted(df['prompt_idx'].unique())
    # Only show fixed values with k=25 curves
    available_fvs = sorted(set(k[1] for k in k25_curves.keys())) if k25_curves else []
    if not available_fvs:
        print(f'No k=25 curves for {agent_name} comparison')
        return None
    
    colors = {'batch_avg': 'tab:blue', 'single_k5': 'tab:red', 'single_k25': 'tab:orange'}
    
    fig, axes = plt.subplots(len(available_fvs), len(prompt_indices),
                             figsize=(6*len(prompt_indices), 4*len(available_fvs)), squeeze=False)
    
    for row, fv in enumerate(available_fvs):
        for col, pidx in enumerate(prompt_indices):
            ax = axes[row, col]
            mask = (df['prompt_idx'] == pidx) & (np.isclose(df[fixed_col], fv))
            subset = df[mask]
            
            if len(subset) == 0:
                ax.set_visible(False)
                continue
            
            # Method 1: Batch-averaged (5 batches of 5) — only k=25 points
            batch_records = []
            for bid_val, grp in subset.groupby(vary_col):
                grp_sorted = grp.sort_values('sample_idx')
                if len(grp_sorted) < TOTAL_SAMPLES:
                    continue
                winners = []
                for bi in range(NUM_BATCHES):
                    batch = grp_sorted.iloc[bi*BATCH_SIZE:(bi+1)*BATCH_SIZE]
                    winners.append(batch.loc[batch['total_welfare'].idxmax(), agent_col])
                batch_records.append((bid_val, np.mean(winners)))
            
            if batch_records:
                br = sorted(batch_records)
                ax.plot([b[0] for b in br], [b[1] for b in br], 'o-',
                       color=colors['batch_avg'], label='Batch avg (5x5)', linewidth=2, markersize=3)
            
            # Method 2: Single welfare-optimal from first 5 (k=5)
            k5_records = []
            for bid_val, grp in subset.groupby(vary_col):
                first5 = grp.sort_values('sample_idx').head(5)
                if len(first5) > 0:
                    best = first5.loc[first5['total_welfare'].idxmax()]
                    k5_records.append((bid_val, best[agent_col]))
            
            if k5_records:
                kr = sorted(k5_records)
                ax.plot([b[0] for b in kr], [b[1] for b in kr], 's--',
                       color=colors['single_k5'], label='Single best (k=5)', linewidth=1, markersize=3, alpha=0.7)
            
            # Method 3: Single welfare-optimal from all 25 (k=25)
            k25_records = []
            for bid_val, grp in subset.groupby(vary_col):
                if len(grp) >= TOTAL_SAMPLES:
                    best = grp.loc[grp['total_welfare'].idxmax()]
                    k25_records.append((bid_val, best[agent_col]))
            
            if k25_records:
                kr25 = sorted(k25_records)
                ax.plot([b[0] for b in kr25], [b[1] for b in kr25], '^--',
                       color=colors['single_k25'], label='Single best (k=25)', linewidth=1, markersize=3, alpha=0.7)
            
            ax.set_xlabel(f'{agent_name} Bid')
            if row == 0:
                ax.set_title(f'P{pidx}', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'Opp={fv}\nCLIP Alignment')
            ax.legend(fontsize=7)
    
    fig.suptitle(f'Method Comparison — {agent_name}', fontsize=14)
    plt.tight_layout()
    return fig

if HAS_DATA and curves_a1:
    fig = compare_methods(df_regular, prompts_regular, 'agent1_alignment', 'b1', 'b2', 'Agent 1', curves_a1)
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'method_comparison_a1.png'), dpi=150, bbox_inches='tight')
    plt.show()

if HAS_DATA and curves_a2:
    fig = compare_methods(df_regular, prompts_regular, 'agent2_alignment', 'b2', 'b1', 'Agent 2', curves_a2)
    if fig:
        fig.savefig(os.path.join(RESULTS_DIR, 'method_comparison_a2.png'), dpi=150, bbox_inches='tight')
    plt.show()

if not curves_a1 and not curves_a2:
    print('No k=25 curves available for comparison')

## 13. Export Data

In [ ]:
dfs_to_concat = []
if len(regret_a1) > 0:
    regret_a1['agent'] = 'Agent 1'
    dfs_to_concat.append(regret_a1)
if len(regret_a2) > 0:
    regret_a2['agent'] = 'Agent 2'
    dfs_to_concat.append(regret_a2)

if dfs_to_concat:
    all_regret = pd.concat(dfs_to_concat, ignore_index=True)
    all_regret.to_csv(os.path.join(RESULTS_DIR, 'data.csv'), index=False)
    print(f'Saved {len(all_regret)} rows to {RESULTS_DIR}/data.csv')
    
    summary = all_regret.groupby('agent').agg(
        mean_regret=('relative_regret', 'mean'),
        max_regret=('relative_regret', 'max'),
        median_regret=('relative_regret', 'median'),
        std_regret=('relative_regret', 'std'),
    ).reset_index()
    print('\nSummary:')
    print(summary.to_string(index=False))
else:
    print('No data to export')